In [2]:
# Basic
!pip install pandas numpy matplotlib scikit-learn torch
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/241.3 MB ? eta -:--:--
   ---------------------------------------- 2.1/241.3 MB 11.8 MB/s eta 0:00:21
    --------------------------------------- 4.5/241.3 MB 11.7 MB/s eta 0:00:21
   - -------------------------------------- 7.1/241.3 MB 11.8 MB/s eta 0:00:20
   - -------------------------------------- 9.4/241.3 MB 11.7 MB/s eta 0:00:20
   - -------------------------------------- 11.8/241.3 MB 11.7 MB/s eta 0:00:20
   -- ------------------------------------- 14.2/241.3 MB 11.7 MB/s eta 0:00:20
   -- ------------------------------------- 16.8/241.3 MB 11.6 MB/s eta 0:00:20
   --- ------------------------------------ 19.1/241.3 MB 11.7 MB/s eta 0:00:19
   --- ------------------------------------ 21.5/241.3 MB 11.7 MB/s eta 0:00:19
   --- ------------------------------------ 23.9/241.3 MB 11.7 MB/s eta 0:00:19
   ---- ----------------------------------- 26.2/241.3 MB 11.8 MB/s eta 0:00:19
   ---- ----------------------------------- 28.8/241.

In [3]:
# Path
train_path = "train_FD001.txt"

# Load train data (handle extra spaces)
df_train = pd.read_csv(train_path, sep="\s+", header=None)
df_train.dropna(axis=1, how='all', inplace=True)

# Assign column names dynamically
n_cols = df_train.shape[1]
df_train.columns = (
    ['engine_unit', 'cycle', 'setting1', 'setting2', 'setting3'] +
    [f'sensor{i}' for i in range(1, n_cols - 5 + 1)]  # 2 IDs + 3 settings
)

print(f"Train shape: {df_train.shape}")
print(df_train.head())



Train shape: (20631, 26)
   engine_unit  cycle  setting1  setting2  setting3  sensor1  sensor2  \
0            1      1   -0.0007   -0.0004     100.0   518.67   641.82   
1            1      2    0.0019   -0.0003     100.0   518.67   642.15   
2            1      3   -0.0043    0.0003     100.0   518.67   642.35   
3            1      4    0.0007    0.0000     100.0   518.67   642.35   
4            1      5   -0.0019   -0.0002     100.0   518.67   642.37   

   sensor3  sensor4  sensor5  ...  sensor12  sensor13  sensor14  sensor15  \
0  1589.70  1400.60    14.62  ...    521.66   2388.02   8138.62    8.4195   
1  1591.82  1403.14    14.62  ...    522.28   2388.07   8131.49    8.4318   
2  1587.99  1404.20    14.62  ...    522.42   2388.03   8133.23    8.4178   
3  1582.79  1401.87    14.62  ...    522.86   2388.08   8133.83    8.3682   
4  1582.85  1406.22    14.62  ...    522.19   2388.04   8133.80    8.4294   

   sensor16  sensor17  sensor18  sensor19  sensor20  sensor21  
0      0.

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Nilay\AppData\Local\Temp\ipykernel_14620\2321325891.py:5: SyntaxWarning: invalid escape sequence '\s'
  df_train = pd.read_csv(train_path, sep="\s+", header=None)


In [4]:
# Max cycle per engine
max_cycles = df_train.groupby('engine_unit')['cycle'].max().reset_index()
max_cycles.columns = ['engine_unit', 'max_cycle']

# Merge to compute RUL
df_train = pd.merge(df_train, max_cycles, on='engine_unit')
df_train['RUL'] = df_train['max_cycle'] - df_train['cycle']
df_train.drop('max_cycle', axis=1, inplace=True)

print(df_train[['engine_unit','cycle','RUL']].head())


   engine_unit  cycle  RUL
0            1      1  191
1            1      2  190
2            1      3  189
3            1      4  188
4            1      5  187


In [5]:
scaler = MinMaxScaler(feature_range=(-1,1))
df_train_scaled = df_train.copy()

cols_to_scale = df_train.columns.difference(['engine_unit', 'cycle', 'RUL'])
df_train_scaled[cols_to_scale] = scaler.fit_transform(df_train[cols_to_scale])

print("Scaled training data ready")


Scaled training data ready


In [6]:
class CMAPSSLSTMDataset(Dataset):
    def __init__(self, df, seq_len=30):
        self.seq_len = seq_len
        self.data = []

        # Group by engine
        for eid, group in df.groupby('engine_unit'):
            group = group.sort_values('cycle')
            values = group.drop(['engine_unit', 'cycle', 'RUL'], axis=1).values
            rul = group['RUL'].values

            for i in range(len(group) - seq_len + 1):
                self.data.append((values[i:i+seq_len], rul[i+seq_len-1]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        X, y = self.data[idx]
        return torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


In [7]:
seq_len = 30
train_dataset = CMAPSSLSTMDataset(df_train_scaled, seq_len=seq_len)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f"Total training sequences: {len(train_dataset)}")


Total training sequences: 17731


In [21]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)      # (batch, seq, hidden)
        out = out[:, -1, :]        # take last timestep
        out = self.fc(out)         # (batch, 1)
        return out.squeeze()


In [22]:
input_size = df_train_scaled.drop(['engine_unit','cycle','RUL'], axis=1).shape[1]
model = LSTMModel(input_size).to(device)  # Move model to device

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 30
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)  # Move batch to device
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(train_loader):.4f}")


Epoch 1/30, Loss: 10443.0882
Epoch 2/30, Loss: 7904.6455
Epoch 3/30, Loss: 6259.8082
Epoch 4/30, Loss: 5197.9416
Epoch 5/30, Loss: 4539.4483
Epoch 6/30, Loss: 4131.4988
Epoch 7/30, Loss: 3955.4901
Epoch 8/30, Loss: 3878.5423
Epoch 9/30, Loss: 3844.2172
Epoch 10/30, Loss: 3829.3208
Epoch 11/30, Loss: 3823.7578
Epoch 12/30, Loss: 2251.9042
Epoch 13/30, Loss: 1612.9918
Epoch 14/30, Loss: 1321.6715
Epoch 15/30, Loss: 1176.7194
Epoch 16/30, Loss: 1087.0304
Epoch 17/30, Loss: 1011.6996
Epoch 18/30, Loss: 966.8686
Epoch 19/30, Loss: 950.0036
Epoch 20/30, Loss: 915.2760
Epoch 21/30, Loss: 916.2763
Epoch 22/30, Loss: 883.2134
Epoch 23/30, Loss: 883.6165
Epoch 24/30, Loss: 852.9789
Epoch 25/30, Loss: 838.2928
Epoch 26/30, Loss: 830.0582
Epoch 27/30, Loss: 800.0877
Epoch 28/30, Loss: 797.1920
Epoch 29/30, Loss: 787.7179
Epoch 30/30, Loss: 767.1677


In [23]:
# Load test data
test_path = "test_FD001.txt"
df_test = pd.read_csv(test_path, sep="\s+", header=None)
df_test.dropna(axis=1, how='all', inplace=True)
df_test.columns = df_train.columns[:-1]  # same as train (no RUL)

# Load true RULs (last cycle)
rul_truth = pd.read_csv("RUL_FD001.txt", sep="\s+", header=None)
rul_truth.columns = ["RUL"]

# Scale sensors/settings
df_test_scaled = df_test.copy()
cols_to_scale = df_test_scaled.columns.difference(['engine_unit','cycle'])
df_test_scaled[cols_to_scale] = scaler.transform(df_test_scaled[cols_to_scale])

# Select last seq_len cycles per engine
last_sequences = []
y_true = []

for i, (eid, group) in enumerate(df_test_scaled.groupby('engine_unit')):
    group = group.sort_values('cycle')
    if len(group) >= seq_len:
        seq = group.drop(['engine_unit','cycle'], axis=1).values[-seq_len:]
        last_sequences.append(seq)
        y_true.append(rul_truth.iloc[i,0])

# Prepare test data
X_test = torch.tensor(np.array(last_sequences), dtype=torch.float32).to(device)  # Move to device
y_true = np.array(y_true, dtype='float32')

print(f"Prepared test sequences: {len(X_test)}")


<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:8: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:8: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Nilay\AppData\Local\Temp\ipykernel_14620\1656402098.py:3: SyntaxWarning: invalid escape sequence '\s'
  df_test = pd.read_csv(test_path, sep="\s+", header=None)
C:\Users\Nilay\AppData\Local\Temp\ipykernel_14620\1656402098.py:8: SyntaxWarning: invalid escape sequence '\s'
  rul_truth = pd.read_csv("RUL_FD001.txt", sep="\s+", header=None)


Prepared test sequences: 100


In [25]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test).cpu().numpy()  # Move output to CPU for numpy

# Success accuracy: % of predictions within ±5 of true RUL
success_mask = np.abs(y_true - y_pred) <= 10
success_accuracy = 100 * np.mean(success_mask)

print(f"Success Accuracy (±5): {success_accuracy:.2f}%")

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

accuracy_per_engine = 100 * (1 - np.abs(y_true - y_pred) / y_true)
accuracy_per_engine = np.clip(accuracy_per_engine, 0, 100)
mean_accuracy = accuracy_per_engine.mean()

print(f"Test MAE: {mae:.2f}")
print(f"Test RMSE: {rmse:.2f}")
print(f"Relative Accuracy: {mean_accuracy:.2f}%")

Success Accuracy (±5): 46.00%
Test MAE: 18.83
Test RMSE: 26.87
Relative Accuracy: 75.75%
